# 02 · Geometry → quality-index labels

Aligns the Tommerup (k=15) geometry campaigns and computes the track
quality index over time at the four sensor locations (m from the **switch-
blade tip**): A1, A2, A7, A11. Switch `cfg.geom_method` between `'std'` and
`'fractal'`.


In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from intelliswitch import Config

# --- paths -----------------------------------------------------------------
# Raw source recordings (~7,800 logs) are large and live on the external drive;
# point DATA_ROOT at them, or symlink/copy the source into data/raw.
DATA_ROOT = Path(r"/Volumes/Seagate Expansion Drive/Mohammad/M Drive DTU/MATLAB Scripts/IntelliswitchScripts")
W10   = DATA_ROOT / "W10v140"       # vibration .log
BLADE = DATA_ROOT / "Bladenr8252"   # geometry .csv

DATA      = Path("../data")                         # project data lake
RAW_CACHE = DATA / "intermediate" / "raw_cache"     # decoded .npz (stage 1)
STORE     = DATA / "processed" / "feature_store"    # extracted PSD features
MODELS    = DATA / "models"                          # trained artifacts

cfg = Config()
cfg.freq_min_hz, cfg.freq_max_hz, cfg.geom_method, cfg.geom_sensor_locations_m

In [ ]:
from intelliswitch.geometry import align_campaigns, build_deg_model

aligned = align_campaigns(BLADE, cfg)
deg = build_deg_model(aligned, cfg)
print('campaigns:', aligned.days.size,
      '| x range (m):', round(float(np.nanmin(aligned.x_blade_m)),1),
      '..', round(float(np.nanmax(aligned.x_blade_m)),1))
tbl = pd.DataFrame(deg.quality.T, columns=list(cfg.sensor_names))
tbl.insert(0, 'days', deg.days)
tbl


### Quality index over time (lower = worse track)


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9,4))
for i, name in enumerate(cfg.sensor_names):
    plt.plot(deg.days, deg.quality[i], '-o', label=name)
plt.xlabel('days before 2019-01-01'); plt.ylabel(f'quality ({cfg.geom_method})')
plt.legend(); plt.grid(True); plt.show()
